# Hyperparameter sweep

Goal:

- Train one model per hyperparameter combination.
- Track every run in the SageMaker MLflow tracking server.
- Export the best model to `s3://<bucket>/trains/models/`.

Grid search tests every combination in a predefined matrix, so each run differs
in exactly one way and the comparison is fair.


## Environment

Install libraries.


In [ ]:
%pip install -q -U ultralytics torch torchvision onnx onnxslim mlflow sagemaker-mlflow

Inspect environment.


In [ ]:
%matplotlib inline

import os
import sys
from pathlib import Path

import boto3
import matplotlib.pyplot as plt
import mlflow
import pandas as pd
import torch

from sagemaker.core.helper.session_helper import Session

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.tracking import tracking_uri

# local dir
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
RUNS = ROOT / "runs"
MODELS = ROOT / "models"

# create dir
for d in (RAW, PROCESSED, RUNS, MODELS):
    d.mkdir(parents=True, exist_ok=True)

REGION = Session().boto_region_name

# get s3 bucket
env_file = Path.home() / ".sagemaker-yolo.env"
if "BUCKET" not in os.environ and env_file.exists():
    for line in env_file.read_text().splitlines():
        key, _, val = line.partition("=")
        os.environ.setdefault(key.strip(), val.strip())

BUCKET = os.environ["BUCKET"]

# bucket keys
S3_RAW = f"s3://{BUCKET}/data/raw"
S3_SPLIT = f"s3://{BUCKET}/data/split"
S3_MODELS = f"s3://{BUCKET}/trains/models"

# device
DEVICE = 0 if torch.cuda.is_available() else "cpu"

# set tracking server
TRACKING_URI = tracking_uri()
mlflow.set_tracking_uri(TRACKING_URI)

# define experiment
EXPERIMENT = "yolo-plate-detection-sweep"
experiment = mlflow.set_experiment(EXPERIMENT)

# get tracking server instance
server = boto3.client("sagemaker").describe_mlflow_tracking_server(
    TrackingServerName=TRACKING_URI.rsplit("/", 1)[-1]
)
UI_URL = server["TrackingServerUrl"]

# print environment
print("torch     ", torch.__version__)
print("cuda      ", torch.cuda.is_available())
print("device    ", DEVICE)
print("bucket    ", BUCKET)
print("tracking  ", TRACKING_URI)
print("experiment", EXPERIMENT, f"(id {experiment.experiment_id})")
print(f"\nUI: {UI_URL}/#/experiments/{experiment.experiment_id}")

## Split data

Download raw data, split, and store in bucket.


In [ ]:
from src.data_loader import build_split, verify_split, write_data_yaml
from src.s3_sync import download, upload

# e.g. 100 for a fast smoke run
LIMIT = 100 
# LIMIT = None  # all images

SPLIT_SEED = 0 # split random seed

# download
print(download(S3_RAW, RAW))

# split
print(build_split(RAW, PROCESSED, val_fraction=0.2, limit=LIMIT, seed=SPLIT_SEED))
print(verify_split(PROCESSED))

# store split
print(upload(PROCESSED, S3_SPLIT, delete=True))

# create data config yaml file
names = (RAW / "classes.txt").read_text().split()
data_yaml = write_data_yaml(ROOT / "configs" / "data.yaml", PROCESSED, names)
print(data_yaml.read_text())

## Define sweep

Define combination of training hyperparameters(`GRID`).


In [ ]:
from itertools import product

from src.data_loader import build_train_cfg

# create training hyperparameters
base_cfg = build_train_cfg(
    device=DEVICE,
    workers=(os.cpu_count() or 2) if DEVICE != "cpu" else 0,
)
base_cfg["project"] = str(ROOT / base_cfg["project"])

# count images
n_images = sum(len(list((PROCESSED / s / "images").iterdir())) for s in ("train", "val"))
tag = "gpu" if DEVICE != "cpu" else "cpu"

# the axes to sweep; add keys here to widen the grid
SWEEP = {
    "epochs": (10, 20, 30),
}

# define grid
GRID = []
for values in product(*SWEEP.values()):
    overrides = dict(zip(SWEEP.keys(), values))
    suffix = "-".join(f"{k}{v}" for k, v in overrides.items())
    overrides["name"] = f"tune-{tag}-{n_images}img-{base_cfg['imgsz']}px-{suffix}"
    GRID.append(overrides)

# print grid
print(f"experiment {EXPERIMENT}")
print(f"{len(GRID)} runs")
for g in GRID:
    print(" ", g)

## Run sweep

Loop each grid entry on its own MLflow run.


In [ ]:
from src.tracking import run_sweep

results = run_sweep(
    grid=GRID,
    base_cfg=base_cfg,
    data_yaml=data_yaml,
    processed_dir=PROCESSED,
    raw_dir=RAW,
    experiment=EXPERIMENT,
    # keep the mlflow run name and the run directory the same
    run_name=lambda cfg: cfg["name"],
)

summary = pd.DataFrame(results)
summary

## Compare runs

Every run in the experiment, side by side.


In [ ]:
from src.tracking import compare_runs

compare_runs(EXPERIMENT)

Plot sweep runs.


In [ ]:
client = mlflow.tracking.MlflowClient()
runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

# only this sweep's runs; the experiment accumulates older ones
wanted = [g["name"] for g in GRID]
runs = runs[runs["params.name"].isin(wanted)]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for _, row in runs.iterrows():
    for ax, metric in zip(axes, ("metrics/mAP50B", "metrics/mAP50-95B")):
        # mlflow does not guarantee ordering, sort by step before plotting
        history = sorted(client.get_metric_history(row["run_id"], metric), key=lambda p: p.step)
        if history:
            ax.plot(
                [p.step for p in history],
                [p.value for p in history],
                marker="o", ms=3, label=row["tags.mlflow.runName"],
            )

for ax, metric in zip(axes, ("mAP50", "mAP50-95")):
    ax.set_xlabel("epoch")
    ax.set_ylabel(metric)
    ax.set_title(metric)
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

### Metrics vs Time: Cost against benefit

More epochs almost always helps a little: **whether the gain justifies the minutes**.

- `gain` and `extra_min` are measured against the cheapest run in the sweep.


In [ ]:
cols = {
    "tags.mlflow.runName": "run",
    "params.epochs": "epochs",
    "metrics.metrics/mAP50-95B": "mAP50-95",
    "metrics.elapsed_seconds": "seconds",
}
cost = runs[[c for c in cols if c in runs.columns]].rename(columns=cols)
cost = cost.dropna(subset=["seconds"]).sort_values("seconds")

# baseline is the cheapest run
base = cost.iloc[0]
cost["minutes"] = (cost["seconds"] / 60).round(1)
cost["gain"] = (cost["mAP50-95"] - base["mAP50-95"]).round(4)
cost["extra_min"] = ((cost["seconds"] - base["seconds"]) / 60).round(1)

# minutes spent per 0.01 mAP gained; NaN for the baseline itself
cost["min_per_0.01"] = (cost["extra_min"] / (cost["gain"] * 100)).round(1)

cost[["run", "epochs", "mAP50-95", "minutes", "gain", "extra_min", "min_per_0.01"]]

## Export the best model

Pick the run with the highest mAP50-95, then export its weights.


In [ ]:
# filter metric
METRIC = "mAP50-95"

# filter completed runs
finished = [r for r in results if "error" not in r]
if not finished:
    # error if all fail.
    raise RuntimeError("every run in the sweep failed")

# sort
ranked = sorted(finished, key=lambda r: r[METRIC], reverse=True)
# get best
best_run = ranked[0]

# print best
print(f"compared {len(finished)} of {len(GRID)} runs\n")
for r in ranked:
    print(f"  {r[METRIC]:.4f}  {r['elapsed_s']:>6}s  {r['name']}")

print(f"\nbest      {best_run['name']}")
print(f"mAP50-95  {best_run[METRIC]:.4f}")
print(f"run_id    {best_run['run_id']}")

Export model.


In [ ]:
import shutil

from ultralytics import YOLO

# get best save dir
save_dir = Path(best_run["save_dir"])
# construct best model
best = YOLO(str(save_dir / "weights" / "best.pt"))

# imgsz must match training
imgsz = base_cfg["imgsz"]
exported = Path(best.export(format="onnx", imgsz=imgsz, opset=12, simplify=True))

onnx_path = MODELS / f"{best_run['name']}.onnx"
shutil.move(str(exported), onnx_path)

print(onnx_path.name)
print(f"{onnx_path.stat().st_size / 1e6:.1f} MB")

Export metadata file.


In [ ]:
import json
from datetime import datetime, timezone

# the graph carries neither class names nor imgsz; the predictor needs both
sidecar = onnx_path.with_suffix(".metadata.json")
sidecar.write_text(json.dumps({
    "imgsz": imgsz,
    "names": [best.names[i] for i in sorted(best.names)],
    "metrics": {
        "mAP50": round(best_run["mAP50"], 4),
        "mAP50-95": round(best_run["mAP50-95"], 4),
    },
    "mlflow": {
        "run_id": best_run["run_id"],
        "experiment": EXPERIMENT,
        "tracking_uri": TRACKING_URI,
    },
    "sweep": {
        "axes": {k: list(v) for k, v in SWEEP.items()},
        "won_with": {k: best_run[k] for k in SWEEP},
        "runs_compared": len(finished),
    },
    "images": n_images,
    "exported_at": datetime.now(timezone.utc).isoformat(),
}, indent=2))

print(sidecar.read_text())

Upload the winner to S3 and attach it to its MLflow run.


In [ ]:
from src.s3_sync import list_objects, upload_files

dest = f"{S3_MODELS}/{best_run['name']}"
weights = save_dir / "weights" / "best.pt"

for uri in upload_files([onnx_path, sidecar, weights], dest):
    print(uri)

# reopen the winning run to attach the export
with mlflow.start_run(run_id=best_run["run_id"]):
    mlflow.log_artifact(str(onnx_path), artifact_path="export")
    mlflow.log_artifact(str(sidecar), artifact_path="export")
    mlflow.set_tag("export.s3_uri", dest)
    mlflow.set_tag("sweep.winner", "true")

print()
for key, meta in sorted(list_objects(dest).items()):
    print(f"  {meta['size'] / 1e6:>6.1f} MB  {key.rsplit('/', 1)[-1]}")

print(f"\nrun: {UI_URL}/#/experiments/{experiment.experiment_id}/runs/{best_run['run_id']}")